# 🌦️ Hava Durumu Uyarı Sistemi

**Amaç:** Hava durumuna (özellikle sıcaklığa) göre çiftçiye hayvan bakımı uyarısı
veren bir sistem kurmak. Örneğin çok sıcakta "sıcaklık stresi riski", çok soğukta
"buzağıları koru" gibi.

**Yaklaşım (iki aşama):**
1. Önce sahte (örnek) hava verisiyle uyarı mantığını kurmak.
2. Sonra ücretsiz bir hava durumu API'sinden (OpenWeatherMap) gerçek veriyi çekip
   aynı mantığı canlı veriye bağlamak.

**Not:** Bu bölüm yapay zeka değil, kural tabanlı bir sistem + veri entegrasyonudur.

In [ ]:
def hava_uyarisi(sicaklik):
    """Sıcaklığa göre hayvan bakımı uyarısı üretir."""
    if sicaklik >= 32:
        return ("🔴 CİDDİ SICAKLIK STRESİ RİSKİ! Hayvanlara bol serin su verin, "
                "gölgelik ve havalandırma sağlayın. Sıcak saatlerde ağır yem vermeyin. "
                "Süt veriminde düşüş görülebilir.")
    elif sicaklik >= 26:
        return ("🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, gölge ve "
                "serinlik sağlayın. Hayvanları sıcak saatlerde gözlemleyin.")
    elif sicaklik <= 0:
        return ("🔵 DON/CİDDİ SOĞUK UYARISI! Özellikle buzağı barınaklarını kontrol "
                "edin, altlığı kuru tutun, su kaplarının donmasını önleyin.")
    elif sicaklik <= 5:
        return ("🟦 Soğuk hava uyarısı. Buzağıları rüzgârdan koruyun, barınağın "
                "kuru ve rüzgârsız olmasına dikkat edin.")
    else:
        return "🟢 Hava koşulları normal. Rutin bakıma devam edebilirsiniz."



test_sicakliklari = [38, 28, 18, 3, -4]

for s in test_sicakliklari:
    print(f"Sıcaklık: {s}°C")
    print("UYARI:", hava_uyarisi(s))
    print("-" * 60)


Sıcaklık: 38°C
UYARI: 🔴 CİDDİ SICAKLIK STRESİ RİSKİ! Hayvanlara bol serin su verin, gölgelik ve havalandırma sağlayın. Sıcak saatlerde ağır yem vermeyin. Süt veriminde düşüş görülebilir.
------------------------------------------------------------
Sıcaklık: 28°C
UYARI: 🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, gölge ve serinlik sağlayın. Hayvanları sıcak saatlerde gözlemleyin.
------------------------------------------------------------
Sıcaklık: 18°C
UYARI: 🟢 Hava koşulları normal. Rutin bakıma devam edebilirsiniz.
------------------------------------------------------------
Sıcaklık: 3°C
UYARI: 🟦 Soğuk hava uyarısı. Buzağıları rüzgârdan koruyun, barınağın kuru ve rüzgârsız olmasına dikkat edin.
------------------------------------------------------------
Sıcaklık: -4°C
UYARI: 🔵 DON/CİDDİ SOĞUK UYARISI! Özellikle buzağı barınaklarını kontrol edin, altlığı kuru tutun, su kaplarının donmasını önleyin.
------------------------------------------------------------


if/elif/else gibi koşullarımla sıcaklık kontrolü yapacağımız ve ona göre uyarı mesajı (bilgilendirme) göndereceğimiz döngülerimizi oluşturduk.Örnek sıcaklık değerleri vererek sistemimizin sahte hava durumu uyarısı yaptıgını denemiş ve görmüş olduk. Ve şimdi beni gittikçe heyecanlandıran o kısma geldik openweathermap den üye olup kendi ekyimi aldım ve gerçek hava durumu analizleriyle uyarı mesajları gönderilmesini sağlicaz umarım çalışır ve harika çalışır.O zaman deneyelim ve görelim.

In [ ]:
import requests

API_KEY = "KEY"   # OpenWeatherMap key bu sayede güncel verileri çekicez.
sehir = "Mugla"

url = f"https://api.openweathermap.org/data/2.5/weather?q={sehir}&appid={API_KEY}&units=metric&lang=tr"

cevap = requests.get(url)
veri = cevap.json()

print("Ham cevap:", veri)

Ham cevap: {'cod': 401, 'message': 'Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.'}


`request` kütüphanem benim OpenWeatherMap sitesine bağlanmak için istek gönderiyor.

`.json `ile de gelen cevabı Python un okuyabileceği hale getiriyor.

In [ ]:
import requests

API_KEY = "KEY"

def hava_durumu_getir(sehir):
    """Gerçek API sürümü — key aktifleşince bu fonksiyonu kullan."""
    url = (f"https://api.openweathermap.org/data/2.5/weather"
           f"?q={sehir},TR&appid={API_KEY}&units=metric&lang=tr")
    veri = requests.get(url).json()

    # API'den gelen veriyi kendi formatımıza çevir
    return {
        "sehir": sehir,
        "sicaklik": round(veri["main"]["temp"]),
        "durum": veri["weather"][0]["description"],
        "nem": veri["main"]["humidity"]
    }


def hava_uyarisi(sicaklik):
    if sicaklik >= 32:
        return ("🔴 CİDDİ SICAKLIK STRESİ RİSKİ! Hayvanlara bol serin su verin, "
                "gölgelik ve havalandırma sağlayın. Sıcak saatlerde ağır yem vermeyin.")
    elif sicaklik >= 26:
        return ("🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, gölge ve "
                "serinlik sağlayın.")
    elif sicaklik <= 0:
        return ("🔵 DON/CİDDİ SOĞUK UYARISI! Buzağı barınaklarını kontrol edin, "
                "altlığı kuru tutun, su kaplarının donmasını önleyin.")
    elif sicaklik <= 5:
        return ("🟦 Soğuk hava uyarısı. Buzağıları rüzgârdan koruyun, barınağın "
                "kuru ve rüzgârsız olmasına dikkat edin.")
    else:
        return "🟢 Hava koşulları normal. Rutin bakıma devam edebilirsiniz."


def hava_paneli(sehir):
    """Hava durumunu getirir ve uyarıyla birlikte gösterir (uygulamadaki 'kart' gibi)."""
    veri = hava_durumu_getir(sehir)
    print("=" * 50)
    print(f"📍 {veri['sehir']}")
    print(f"🌡️  Sıcaklık: {veri['sicaklik']}°C")
    print(f"☁️  Durum: {veri['durum']}")
    print(f"💧 Nem: %{veri['nem']}")
    print("-" * 50)
    print("⚠️ HAYVAN BAKIM UYARISI:")
    print(hava_uyarisi(veri["sicaklik"]))
    print("=" * 50)


# Test
hava_paneli("Denizli")

📍 Denizli
🌡️  Sıcaklık: 29°C
☁️  Durum: açık
💧 Nem: %24
--------------------------------------------------
⚠️ HAYVAN BAKIM UYARISI:
🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, gölge ve serinlik sağlayın.


Şimdi OpenWeatherMap den aldııgmız key henüz aktifleşmedi şuan burada yaptıgımız sahte bi veriyle yolumuza devam edip vakit kaybetmemek.Keyimiz aktifleşince bu sahte veri kısmını silip gerçek orjinal verilerimizi entegre edicez.


Evetttt tam da istrediğimiz gibi keyimiz aktifleşti ve güncel hava durumunu openweathermap den güncel olarak alıyor ve bize uyarı mesajı atıyor gerçekten cok tatlı olduuu  :))


şimdi de çiftçimiz bi anda başka şehire bakmak istedi ve şehir değiştirmek istedi bunun kodunu yazmaya çalışalım. öncelikle başka şehirler için de çalışıyor mu kontrol edelim

In [ ]:
hava_paneli("Istanbul")
print()
hava_paneli("Erzurum")
print()
hava_paneli("Antalya")

📍 Istanbul
🌡️  Sıcaklık: 26°C
☁️  Durum: açık
💧 Nem: %100
--------------------------------------------------
⚠️ HAYVAN BAKIM UYARISI:
🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, gölge ve serinlik sağlayın.

📍 Erzurum
🌡️  Sıcaklık: 25°C
☁️  Durum: açık
💧 Nem: %32
--------------------------------------------------
⚠️ HAYVAN BAKIM UYARISI:
🟢 Hava koşulları normal. Rutin bakıma devam edebilirsiniz.

📍 Antalya
🌡️  Sıcaklık: 30°C
☁️  Durum: açık
💧 Nem: %58
--------------------------------------------------
⚠️ HAYVAN BAKIM UYARISI:
🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, gölge ve serinlik sağlayın.


In [ ]:
import requests

API_KEY = "KEY"

def hava_durumu_getir(sehir):
    """Gerçek API'den veri çeker. Şehir bulunamazsa None döner."""
    url = (f"https://api.openweathermap.org/data/2.5/weather"
           f"?q={sehir},TR&appid={API_KEY}&units=metric&lang=tr")
    veri = requests.get(url).json()

    # API hata kodu döndürdüyse (şehir bulunamadı vb.)
    if veri.get("cod") != 200:
        return None

    return {
        "sehir": sehir,
        "sicaklik": round(veri["main"]["temp"]),
        "durum": veri["weather"][0]["description"],
        "nem": veri["main"]["humidity"],
        "ruzgar": veri["wind"]["speed"]
    }


def hava_paneli(sehir):
    veri = hava_durumu_getir(sehir)

    # Şehir bulunamadıysa kullanıcıyı bilgilendir, çökme
    if veri is None:
        print(f"❌ '{sehir}' için hava durumu bulunamadı.")
        print("Lütfen şehir adını kontrol edin (örn: Denizli, Istanbul, Antalya).")
        return

    print("=" * 50)
    print(f"📍 {veri['sehir']}")
    print(f"🌡️  Sıcaklık: {veri['sicaklik']}°C")
    print(f"☁️  Durum: {veri['durum']}")
    print(f"💧 Nem: %{veri['nem']}")
    print(f"💨 Rüzgar: {veri['ruzgar']} m/s")
    print("-" * 50)
    print("⚠️ HAYVAN BAKIM UYARISI:")
    print(hava_uyarisi(veri["sicaklik"]))
    print("=" * 50)


# Test — biri gerçek, biri hatalı şehir
hava_paneli("Denizli")
print()
hava_paneli("asdfgh")   # hatalı şehir, çökmemeli

📍 Denizli
🌡️  Sıcaklık: 29°C
☁️  Durum: açık
💧 Nem: %24
💨 Rüzgar: 1.47 m/s
--------------------------------------------------
⚠️ HAYVAN BAKIM UYARISI:
🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, gölge ve serinlik sağlayın.

❌ 'asdfgh' için hava durumu bulunamadı.
Lütfen şehir adını kontrol edin (örn: Denizli, Istanbul, Antalya).


genel bi düzenleme yapıyoruz .Kontrollerimizi ekledik .Hatalı şehir girişi varsa sistemin çökmesini engelleyip uyarı döndürmesini sağladık şimdi de mesajlarımızı ve uyarılarımızı zenginleştirelim ki gelen mesajı kullanıcılar gördüklerinde "vayy beeee" desinlerr hadi başlayalım.


In [ ]:
def hava_uyarisi(veri):
    """Sıcaklık, nem ve rüzgara göre çok boyutlu uyarı üretir."""
    sicaklik = veri["sicaklik"]
    nem = veri["nem"]
    durum = veri["durum"].lower()

    uyarilar = []

    # Sıcaklık kaynaklı
    if sicaklik >= 32:
        uyarilar.append("🔴 CİDDİ SICAKLIK STRESİ! Bol serin su, gölge ve "
                        "havalandırma sağlayın; sıcak saatlerde ağır yem vermeyin.")
    elif sicaklik >= 26:
        uyarilar.append("🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, "
                        "gölge ve serinlik sağlayın.")
    elif sicaklik <= 0:
        uyarilar.append("🔵 DON UYARISI! Buzağı barınaklarını kontrol edin, "
                        "su kaplarının donmasını önleyin.")
    elif sicaklik <= 5:
        uyarilar.append("🟦 Soğuk hava. Buzağıları rüzgârdan koruyun, barınağı "
                        "kuru ve rüzgârsız tutun.")

    # Nem + sıcaklık birlikte (sıcak + nemli = daha tehlikeli)
    if sicaklik >= 26 and nem >= 70:
        uyarilar.append("💧 Yüksek nem sıcaklık stresini ağırlaştırır! "
                        "Havalandırmaya ekstra dikkat edin.")

    # Yağış durumu
    if "yağmur" in durum or "sağanak" in durum:
        uyarilar.append("🌧️ Yağışlı hava. Zeminin ıslanması ayak/tırnak "
                        "hastalıklarına yol açabilir; hayvanları kuru alanda tutun.")

    # Hiç uyarı yoksa
    if not uyarilar:
        return "🟢 Hava koşulları normal. Rutin bakıma devam edebilirsiniz."

    return "\n".join(uyarilar)

In [ ]:
def hava_uyarisi(veri):
    """Sıcaklık, nem ve yağışa göre çok boyutlu uyarı üretir."""
    sicaklik = veri["sicaklik"]
    nem = veri["nem"]
    durum = veri["durum"].lower()

    uyarilar = []

    if sicaklik >= 32:
        uyarilar.append("🔴 CİDDİ SICAKLIK STRESİ! Bol serin su, gölge ve "
                        "havalandırma sağlayın; sıcak saatlerde ağır yem vermeyin.")
    elif sicaklik >= 26:
        uyarilar.append("🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, "
                        "gölge ve serinlik sağlayın.")
    elif sicaklik <= 0:
        uyarilar.append("🔵 DON UYARISI! Buzağı barınaklarını kontrol edin, "
                        "su kaplarının donmasını önleyin.")
    elif sicaklik <= 5:
        uyarilar.append("🟦 Soğuk hava. Buzağıları rüzgârdan koruyun, barınağı "
                        "kuru ve rüzgârsız tutun.")

    if sicaklik >= 26 and nem >= 70:
        uyarilar.append("💧 Yüksek nem sıcaklık stresini ağırlaştırır! "
                        "Havalandırmaya ekstra dikkat edin.")

    if "yağmur" in durum or "sağanak" in durum:
        uyarilar.append("🌧️ Yağışlı hava. Islak zemin ayak/tırnak hastalıklarına "
                        "yol açabilir; hayvanları kuru alanda tutun.")

    if not uyarilar:
        return "🟢 Hava koşulları normal. Rutin bakıma devam edebilirsiniz."

    return "\n".join(uyarilar)


def hava_paneli(sehir):
    veri = hava_durumu_getir(sehir)

    if veri is None:
        print(f"❌ '{sehir}' için hava durumu bulunamadı.")
        print("Lütfen şehir adını kontrol edin (örn: Denizli, Istanbul, Antalya).")
        return

    print("=" * 50)
    print(f"📍 {veri['sehir']}")
    print(f"🌡️  Sıcaklık: {veri['sicaklik']}°C")
    print(f"☁️  Durum: {veri['durum']}")
    print(f"💧 Nem: %{veri['nem']}")
    print("-" * 50)
    print("⚠️ HAYVAN BAKIM UYARISI:")
    print(hava_uyarisi(veri))
    print("=" * 50)


# Test
hava_paneli("Denizli")

📍 Denizli
🌡️  Sıcaklık: 29°C
☁️  Durum: açık
💧 Nem: %24
--------------------------------------------------
⚠️ HAYVAN BAKIM UYARISI:
🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, gölge ve serinlik sağlayın.


In [ ]:
hava_paneli("Rize")      # muhtemelen nemli/yağışlı, farklı uyarılar çıkabilir
print()
hava_paneli("Trabzon")   # sahil, nem yüksek olabilir

📍 Rize
🌡️  Sıcaklık: 22°C
☁️  Durum: açık
💧 Nem: %57
--------------------------------------------------
⚠️ HAYVAN BAKIM UYARISI:
🟢 Hava koşulları normal. Rutin bakıma devam edebilirsiniz.

📍 Trabzon
🌡️  Sıcaklık: 23°C
☁️  Durum: açık
💧 Nem: %76
--------------------------------------------------
⚠️ HAYVAN BAKIM UYARISI:
🟢 Hava koşulları normal. Rutin bakıma devam edebilirsiniz.


In [ ]:
# Çift uyarıyı garanti görmek için sahte veri (sıcak + nemli)
sahte = {"sehir": "Test", "sicaklik": 30, "durum": "açık", "nem": 75}
print("SAHTE TEST (30°C, %75 nem):")
print(hava_uyarisi(sahte))

SAHTE TEST (30°C, %75 nem):
🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, gölge ve serinlik sağlayın.
💧 Yüksek nem sıcaklık stresini ağırlaştırır! Havalandırmaya ekstra dikkat edin.


şimdi hava durumuyla alakalı bi sıkıntımız kalmadı.amacım suanda rag sistemini entegre etmek ve rag asistanımla birlikte güzel bi sistem hazırlamak .e tbai bunun için de her şeyi buraya entegre etmemiz gerekiyo ama basaracağız. hadi baslayalım


In [ ]:
!pip install sentence-transformers -q

In [ ]:
# Asistan parçaları hazır mı kontrol et
try:
    print("embed_model:", "VAR" if 'embed_model' in dir() else "YOK")
    print("bilgi_tabani:", len(bilgi_tabani), "konu" if 'bilgi_tabani' in dir() else "YOK")
    print("bilgi_embed:", "VAR" if 'bilgi_embed' in dir() else "YOK")
    print("llm:", "VAR" if 'llm' in dir() else "YOK")
    print("veteriner_asistan:", "VAR" if 'veteriner_asistan' in dir() else "YOK")
except Exception as e:
    print("Eksik olan var:", e)

embed_model: YOK
Eksik olan var: name 'bilgi_tabani' is not defined


In [ ]:
from sentence_transformers import SentenceTransformer, util

embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

bilgi_tabani = [
    "Buzağıların yem yememesinin yaygın sebepleri geçiş dönemi stresi, sindirim bozuklukları, diş çıkarma ağrısı, soğuk veya kirli su, ani yem değişikliği ve bağırsak parazitleridir. Yem değişikliği kademeli yapılmalı, temiz su sağlanmalı ve ortam sıcak tutulmalıdır. İştahsızlık iki günü geçerse veteriner hekime başvurulmalıdır.",
    "Süt veriminin düşmesinin başlıca nedenleri yetersiz veya kalitesiz yem, su tüketiminin azalması, sıcaklık stresi, meme iltihabı (mastitis), laktasyon döneminin ilerlemesi ve gebeliktir. İlk kontrol edilmesi gerekenler yem kalitesi, yem miktarı ve temiz suya erişimdir. Ani düşüşte meme mastitis açısından kontrol edilmeli, sorun sürerse veteriner hekime danışılmalıdır.",
    "İneklerde topallık genellikle tırnak hastalıkları (çürük, çatlak), sert veya sürekli ıslak zemin, eklem iltihabı, yaralanma ve mineral eksikliğinden kaynaklanır. Topallayan hayvanın yem tüketimi ve süt verimi düşebilir. Erken fark edilirse tırnak bakımı ve zemin düzenlemesiyle tedavi başarısı yüksektir; ilerlemiş durumlarda veteriner müdahalesi gerekir.",
    "Gebe ineklerin doğuma yakın kuru döneminde beslenmesi kritiktir. Enerji ve protein ihtiyacı artar; ancak aşırı besleme doğum güçlüğüne ve metabolik hastalıklara yol açabilir. Doğumdan yaklaşık üç hafta önce geçiş rasyonuna başlanmalı, mineral ve vitamin desteği verilmelidir. Doğum yaklaştığında hayvan temiz ve rahat bir bölmede gözlem altında tutulmalıdır.",
    "Aşı takvimine uyulması hastalıkların önlenmesinin temelidir. Şap, brusella, şarbon ve yanıkara gibi hastalıklara karşı düzenli aşılama yapılmalıdır. Aşı zamanları hayvanın yaşına ve bölgedeki risklere göre veteriner hekim tarafından belirlenir. Aşı sonrası hayvan birkaç gün gözlem altında tutulmalı ve aşı kayıtları düzenli olarak tutulmalıdır.",
    "Süt sağımı hijyeni mastitis (meme iltihabı) riskini azaltmanın en önemli yoludur. Sağımdan önce meme temizlenip kurulanmalı, sağım ekipmanları her kullanımdan sonra dezenfekte edilmelidir. İlk sütte pıhtı, kan veya renk değişikliği görülürse mastitis şüphesiyle veteriner hekime danışılmalıdır. Sağım sonrası memelerin temiz ortamda tutulması enfeksiyon riskini düşürür.",
    "İshal, özellikle buzağılarda tehlikelidir ve hızlı sıvı kaybına yol açar. Nedenleri arasında bakteri, virüs veya parazit enfeksiyonları, ani yem değişikliği, kirli su ve hijyen eksikliği bulunur. Hayvana bol temiz su ve gerekirse elektrolit verilmelidir. İshal bir günden uzun sürerse, kanlıysa veya hayvan halsizse acilen veteriner hekime başvurulmalıdır.",
    "Yeterli ve temiz su, hayvan sağlığı ve süt verimi için kritiktir. Bir süt ineği günde yaklaşık 60-100 litre su içebilir; su kısıtlandığında süt verimi hızla düşer. Su kaynağı temiz, kolay erişilebilir ve sıcak havalarda serin tutulmalıdır. Kirli veya yetersiz su iştahsızlık ve hastalıklara zemin hazırlar.",
    "İç ve dış parazitler hayvanlarda zayıflama, kıl dökülmesi, kaşıntı, kansızlık ve verim düşüklüğüne yol açar. İç parazitler için ilaçlama, dış parazitler için uygun uygulamalar düzenli olarak yapılmalıdır. Parazit kontrol programı, mevsime ve bölge koşullarına göre veteriner hekim önerisiyle planlanmalıdır.",
    "Doğum sırasında hayvan sakin, temiz ve gözlem altında tutulmalıdır. Normal doğum genellikle birkaç saat içinde tamamlanır. Doğum uzarsa, buzağının duruşu ters ise veya hayvan aşırı zorlanıyorsa vakit kaybetmeden veteriner hekim çağrılmalıdır. Doğum sonrası hem ana hem yavru yakından izlenmelidir.",
    "Yeni doğan buzağının ilk saatlerde ağız sütü (kolostrum) alması hayati önemdedir. Kolostrum buzağıya bağışıklık kazandırır ve hastalıklara karşı korur. İlk iki saat içinde, doğum ağırlığının yaklaşık yüzde onu kadar kolostrum verilmelidir. Geciken veya yetersiz kolostrum, buzağının hastalanma ve ölüm riskini belirgin artırır.",
    "Sıcaklık stresi, özellikle yaz aylarında süt ineklerinde verim düşüşüne, iştahsızlığa ve solunum hızlanmasına yol açar. Hayvanlara gölgelik, iyi havalandırma ve bol serin su sağlanmalıdır. Sıcak saatlerde ağır yem yerine daha hafif ve sindirilebilir beslenme tercih edilmelidir. Aşırı sıcak hayvan sağlığı için ciddi bir risktir.",
    "Hayvanların sağlıklı olması ve verimli çalışması için dengeli bir rasyon şarttır. Rasyon; enerji, protein, lif, mineral ve vitaminleri hayvanın ihtiyacına göre içermelidir. Kaba yem (ot, silaj) ve kesif yem (tahıl karması) dengesi önemlidir. Dengesiz besleme, verim düşüklüğüne ve sindirim sorunlarına yol açar.",
    "Şişkinlik (timpani), işkembede aşırı gaz birikmesiyle oluşan, hızlı gelişebilen ciddi bir durumdur. Genellikle aşırı taze veya yaş yonca gibi baklagillerin fazla tüketilmesiyle görülür. Hayvanın sol böğrü belirgin şişer, huzursuzluk ve solunum güçlüğü olur. Şişkinlik acil bir durumdur; derhal veteriner hekime başvurulmalıdır.",
    "Düzenli tırnak bakımı topallığı ve ayak hastalıklarını önlemenin temelidir. Tırnaklar aşırı uzadığında hayvanın duruşu bozulur ve yürüme güçleşir. Ahır zemini kuru, temiz ve kaymayı önleyecek şekilde olmalıdır. Yılda birkaç kez tırnak kesimi ve kontrolü önerilir; belirgin aksama varsa veteriner hekime danışılmalıdır.",
    "Kızgınlık (östrus) belirtilerinin doğru takibi, başarılı tohumlama için gereklidir. Belirtiler arasında huzursuzluk, diğer hayvanlara atlama veya atlanmaya izin verme, iştah değişikliği ve akıntı bulunur. Kızgınlık genellikle belirli aralıklarla tekrarlar. Doğru zamanda tohumlama için kızgınlık günü kayıt altına alınmalıdır.",
    "Buzağılar bağışıklıkları zayıf olduğu için temiz, kuru ve rüzgârdan korunaklı bir barınakta tutulmalıdır. Islak ve kirli zemin, ishal ve solunum hastalıklarına davetiye çıkarır. Barınak düzenli temizlenmeli, altlık kuru tutulmalı ve yeterli temiz hava sağlanmalıdır. Hasta buzağılar sağlıklı olanlardan ayrılmalıdır.",
    "Solunum yolu hastalıkları, özellikle genç hayvanlarda öksürük, burun akıntısı, hızlı solunum ve ateşle kendini gösterir. Nedenleri arasında soğuk, nemli ve kötü havalandırılan barınaklar, ani sıcaklık değişimleri ve enfeksiyonlar bulunur. Erken fark edilirse tedavi başarılıdır; belirtiler görülürse veteriner hekime başvurulmalıdır.",
    "Mineral ve vitamin eksiklikleri; iştahsızlık, zayıflama, tüy ve kıl bozuklukları, üreme sorunları ve verim düşüklüğüne yol açabilir. Özellikle kalsiyum, fosfor, selenyum ve A, D, E vitaminleri önemlidir. Dengeli rasyon ve gerektiğinde mineral takviyesiyle önlenebilir. Takviye programı veteriner hekim önerisiyle belirlenmelidir.",
    "Süt humması (doğum felci), genellikle doğumdan hemen sonra kandaki kalsiyumun ani düşmesiyle görülür. Hayvan halsizleşir, ayağa kalkamaz ve titreme görülebilir. Özellikle yüksek verimli ve yaşlı ineklerde risk daha yüksektir. Bu acil bir durumdur; derhal veteriner hekime başvurulmalıdır.",
    "Yemdeki ani değişiklikler işkembe dengesini bozarak sindirim sorunlarına, iştahsızlığa ve verim düşüklüğüne yol açar. Yeni bir yeme geçiş birkaç gün içinde kademeli yapılmalıdır. İşkembe sağlığı için yeterli kaba yem (lif) verilmesi önemlidir. Ani ve aşırı kesif yem, asidoz gibi sorunlara neden olabilir.",
    "Hayvanların günlük gözlemi, sorunların erken fark edilmesini sağlar. İştah, hareketlilik, dışkı kıvamı, süt verimi ve davranıştaki değişimler önemli göstergelerdir. Kulakların düşük olması, sürüden ayrı durma, iştahsızlık veya verim düşüşü dikkat edilmesi gereken işaretlerdir. Şüpheli durumlarda veteriner hekime danışılmalıdır.",
    "Yem ve su kaplarının düzenli temizliği hastalıkların yayılmasını önler. Kirli kaplarda bakteri ve küf üreyebilir; bu da iştahsızlık ve sindirim sorunlarına yol açar. Su kapları her gün, yemlikler düzenli aralıklarla temizlenmelidir. Küflenmiş veya bozulmuş yem kesinlikle hayvana verilmemelidir.",
    "Sürüye yeni katılan hayvanlar, hastalık taşıma riskine karşı bir süre karantinada (ayrı) tutulmalıdır. Bu sürede hayvan gözlemlenmeli, gerekli aşı ve parazit kontrolleri yapılmalıdır. Karantina, bulaşıcı hastalıkların tüm sürüye yayılmasını önleyen önemli bir koruyucu tedbirdir. Şüpheli belirtilerde veteriner hekime danışılmalıdır."
]

bilgi_embed = embed_model.encode(bilgi_tabani, convert_to_tensor=True)
print("Bilgi tabanı hazır:", len(bilgi_tabani), "konu")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Bilgi tabanı hazır: 24 konu


In [ ]:
from transformers import pipeline
import torch

llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Dil modeli yüklendi.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Dil modeli yüklendi.


In [ ]:
def konu_uygun_mu(soru):
    kontrol = [
        {"role": "system", "content":
         "Görevin: verilen sorunun hayvan/çiftlik/veteriner konusuyla ilgili olup "
         "olmadığına karar vermek. İlgiliyse 'EVET', değilse 'HAYIR' yaz.\n"
         "Örnekler:\n"
         "'inek neden süt vermiyor' -> EVET\n"
         "'buzağıda ishal var' -> EVET\n"
         "'sıcak havada hayvana ne yapmalıyım' -> EVET\n"
         "'hava durumu nasıl' -> HAYIR\n"
         "'telefonum bozuldu' -> HAYIR\n"
         "Sadece EVET veya HAYIR yaz."},
        {"role": "user", "content": soru}
    ]
    cevap = llm(kontrol, max_new_tokens=5, do_sample=False)
    return "EVET" in cevap[0]["generated_text"][-1]["content"].upper()


def veteriner_asistan(soru):
    if not konu_uygun_mu(soru):
        return ("Ben bir veteriner asistanıyım ve yalnızca hayvan sağlığı ile ilgili "
                "sorulara yardımcı olabilirim.")

    soru_embed = embed_model.encode(soru, convert_to_tensor=True)
    benzerlikler = util.cos_sim(soru_embed, bilgi_embed)[0]
    en_iyi_index = benzerlikler.argsort(descending=True)[:2]
    secili_bilgiler = "\n\n".join([bilgi_tabani[i] for i in en_iyi_index])

    mesaj = [
        {"role": "system", "content":
         "Sen bir veteriner asistanısın. SADECE sana verilen bilgilere dayanarak "
         "cevap ver. Soruyla EN ALAKALI bilgiyi kullan. Bilgide olmayanı uydurma. "
         "Teşhis koyma ve sonunda mutlaka veteriner hekime danışılmasını öner."},
        {"role": "user", "content":
         f"Bilgiler:\n{secili_bilgiler}\n\nSoru: {soru}"}
    ]
    cevap = llm(mesaj, max_new_tokens=200, do_sample=False)
    return cevap[0]["generated_text"][-1]["content"]


print("Asistan hazır.")

Asistan hazır.


In [ ]:
import requests

API_KEY = "f2de4dd63b178dfdf33b2f546ee65b03"

def hava_durumu_getir(sehir):
    """Gerçek API'den veri çeker. Şehir bulunamazsa None döner."""
    url = (f"https://api.openweathermap.org/data/2.5/weather"
           f"?q={sehir},TR&appid={API_KEY}&units=metric&lang=tr")
    veri = requests.get(url).json()

    # API hata kodu döndürdüyse (şehir bulunamadı vb.)
    if veri.get("cod") != 200:
        return None

    return {
        "sehir": sehir,
        "sicaklik": round(veri["main"]["temp"]),
        "durum": veri["weather"][0]["description"],
        "nem": veri["main"]["humidity"] # 'ruzgar' is not used in the problem description here, so leaving it out for brevity and consistency
    }

def hava_uyarisi(veri):
    """Sıcaklık, nem ve yağışa göre çok boyutlu uyarı üretir."""
    sicaklik = veri["sicaklik"]
    nem = veri["nem"]
    durum = veri["durum"].lower()

    uyarilar = []

    if sicaklik >= 32:
        uyarilar.append("🔴 CİDDİ SICAKLIK STRESİ! Bol serin su, gölge ve "
                        "havalandırma sağlayın; sıcak saatlerde ağır yem vermeyin.")
    elif sicaklik >= 26:
        uyarilar.append("🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, "
                        "gölge ve serinlik sağlayın.")
    elif sicaklik <= 0:
        uyarilar.append("🔵 DON UYARISI! Buzağı barınaklarını kontrol edin, "
                        "su kaplarının donmasını önleyin.")
    elif sicaklik <= 5:
        uyarilar.append("🟦 Soğuk hava. Buzağıları rüzgârdan koruyun, barınağı "
                        "kuru ve rüzgârsız tutun.")

    if sicaklik >= 26 and nem >= 70:
        uyarilar.append("💧 Yüksek nem sıcaklık stresini ağırlaştırır! "
                        "Havalandırmaya ekstra dikkat edin.")

    if "yağmur" in durum or "sağanak" in durum:
        uyarilar.append("🌧️ Yağışlı hava. Islak zemin ayak/tırnak hastalıklarına "
                        "yol açabilir; hayvanları kuru alanda tutun.")

    if not uyarilar:
        return "🟢 Hava koşulları normal. Rutin bakıma devam edebilirsiniz."

    return "\n".join(uyarilar)

In [ ]:
def akilli_hava_danismani(sehir):
    """Hava verisini çeker, duruma göre asistana danışır."""
    veri = hava_durumu_getir(sehir)

    if veri is None:
        print(f"❌ '{sehir}' bulunamadı.")
        return

    # Hava panelini göster
    print("=" * 55)
    print(f"📍 {veri['sehir']}  |  🌡️ {veri['sicaklik']}°C  |  "
          f"☁️ {veri['durum']}  |  💧 %{veri['nem']}")
    print("-" * 55)

    # Kural tabanlı uyarı
    print("⚠️ HIZLI UYARI:")
    print(hava_uyarisi(veri))
    print("-" * 55)

    # Havaya göre asistana danış (ZİRVE: hava + RAG asistanı)
    if veri["sicaklik"] >= 26:
        soru = f"Hava {veri['sicaklik']} derece ve sıcak, hayvanlarımı sıcaktan korumak için ne yapmalıyım?"
    elif veri["sicaklik"] <= 5:
        soru = f"Hava {veri['sicaklik']} derece ve soğuk, hayvanlarımı ve buzağıları soğuktan korumak için ne yapmalıyım?"
    else:
        soru = "Hava normal seyrediyor, hayvanlarımın günlük bakımında nelere dikkat etmeliyim?"

    print("🤖 ASİSTAN TAVSİYESİ:")
    print(f"(Sorulan: {soru})\n")
    print(veteriner_asistan(soru))
    print("=" * 55)


# 3 ÖRNEK — farklı hava koşulları
akilli_hava_danismani("Denizli")
print("\n\n")
akilli_hava_danismani("Erzurum")
print("\n\n")
akilli_hava_danismani("Antalya")

[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📍 Denizli  |  🌡️ 32°C  |  ☁️ açık  |  💧 %33
-------------------------------------------------------
⚠️ HIZLI UYARI:
🔴 CİDDİ SICAKLIK STRESİ! Bol serin su, gölge ve havalandırma sağlayın; sıcak saatlerde ağır yem vermeyin.
-------------------------------------------------------
🤖 ASİSTAN TAVSİYESİ:
(Sorulan: Hava 32 derece ve sıcak, hayvanlarımı sıcaktan korumak için ne yapmalıyım?)



[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hava 32 derece ve sıcaktır, bu durumda süt ineklerinizin sıcak stresinden kurtulmak için aşağıdaki adımları izlemenizi öneririm:

1. Hayvanların gölgelik konumlarına getirmeyi unutmayın. Bu, sıcak stresden korunmayı sağlar.
2. İyi havalandırma sağlamak için suyun sürekli olarak erimesini sağlayın. Bu, solunum hızlanmasına engel olabilir.
3. Sıcak saatlerde daha hafif ve sindirilebilir yemlerden ve su almasını sağlayın. Bu, enerji tüketiminin azaldığını ve verimliliğin artıracağını sağlar.
4. Hayvanların karantinada tutulması ve gerekli aşı ve parazit kontrollerinin yapılması önemlidir. Bu, bulaşıcı hastalıkların



📍 Erzurum  |  🌡️ 25°C  |  ☁️ açık  |  💧 %27
-------------------------------------------------------
⚠️ HIZLI UYARI:
🟢 Hava koşulları normal. Rutin bakıma devam edebilirsiniz.
-------------------------------------------------------
🤖 ASİSTAN TAVSİYESİ:
(Sorulan: Hava normal seyrediyor, hayvanlarımın günlük bakımında nelere dikkat etmeliyim?)



[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hava normalseverken, hayvanlarınızın günlük bakımında dikkat etmeniz gereken noktalar şunlardır:

1. İştah: Hayvanların iştahlı olup olmadığını kontrol edin. Düşük iştah, bir sorunun başlamasına neden olabilir.

2. Hareketlilik: Hayvanların normal seviyede mi hareket ettiğini kontrol edin. Eğer hareketliliği azaltmışsa, bu bir uyarı belirtisi olabilir.

3. Dışkı kıvamı: Dışkı sıvının normal seviyede mi olduğunu kontrol edin. Dışkı sıvının düşük olması, bir sindirim sorunu olabileceğini gösterir.

4. Süt verimi: Hayvanların süt verimini kontrol edin. Eğer verim düşmüşse, bu bir sorunun başlamasına neden



📍 Antalya  |  🌡️ 30°C  |  ☁️ açık  |  💧 %61
-------------------------------------------------------
⚠️ HIZLI UYARI:
🟠 Sıcaklık stresi başlayabilir. Su tüketimini artırın, gölge ve serinlik sağlayın.
-------------------------------------------------------
🤖 ASİSTAN TAVSİYESİ:
(Sorulan: Hava 30 derece ve sıcak, hayvanlarımı sıcaktan korumak için ne yapmalıyım?)



[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hava 30 derece ve sıcak olduğu için hayvanların sıcaktan korunması önemlidir. Bilgilerimiz sayesinde şu adımları atabilirsiniz:

1. Hayvanların gölgelik konumlarına getirin. Bu, sıcak hava altında kalan hayvanların ısıya açık kalmasını önlemeye yardımcı olur.

2. Hayvanların iyi havalandırma sağlayalım. Bu, sıcak hava altında kalan hayvanların nefes almayı kolaylaştırır.

3. Bol suyu verin. Solunum hızlanmasına neden olan sıcak hava altında, hayvanların daha fazla su tüketmesi ve solunum sağlamak için bol suyu vermelidir.

4. Hayvanların yemini daha hafif ve sindirilebilir yapın. Aşırı sıcak hava altında, hayvanların daha az yem yeme


In [ ]:
def konu_uygun_mu(soru):
    kontrol = [
        {"role": "system", "content":
         "Görevin: verilen sorunun hayvan/çiftlik/veteriner konusuyla ilgili olup "
         "olmadığına karar vermek. Soru HAYVANLARLA ilgiliyse (hava, mevsim gibi "
         "kelimeler geçse bile) 'EVET' yaz. Sadece hava/genel bilgi soruluyorsa 'HAYIR'.\n"
         "Örnekler:\n"
         "'inek neden süt vermiyor' -> EVET\n"
         "'buzağıda ishal var' -> EVET\n"
         "'hava sıcak, hayvanlarımı nasıl korurum' -> EVET\n"
         "'soğukta buzağıya ne yapmalıyım' -> EVET\n"
         "'hava durumu nasıl olacak' -> HAYIR\n"
         "'telefonum bozuldu' -> HAYIR\n"
         "Sadece EVET veya HAYIR yaz."},
        {"role": "user", "content": soru}
    ]
    cevap = llm(kontrol, max_new_tokens=5, do_sample=False)
    return "EVET" in cevap[0]["generated_text"][-1]["content"].upper()

## 🎯 Kapanış — Hava Durumu + Asistan Entegrasyonu

Bu notebook'ta, gerçek hava durumu verisini yapay zeka asistanıyla birleştiren
bütünleşik bir sistem kurduk.

**Yapılanlar**
1. OpenWeatherMap API'sinden gerçek, canlı hava verisi çekme (sıcaklık, nem, durum)
2. Hatalı şehir girişini düzgün yönetme (çökme yerine bilgilendirme)
3. Sıcaklık + nem + yağışa göre çok boyutlu kural tabanlı uyarı üretme
4. RAG veteriner asistanını bu notebook'a entegre etme
5. Havaya göre asistana otomatik danışma: sistem hava durumuna uygun soruyu
   asistana sorup, bilgi tabanından detaylı bakım tavsiyesi üretiyor

**Sonuç**
Çiftçi bir şehir girdiğinde sistem; o şehrin gerçek havasını gösteriyor, hızlı bir
uyarı veriyor ve yapay zeka asistanıyla havaya özel bakım tavsiyesi sunuyor. Üç
farklı bileşen (veri servisi + kural motoru + RAG asistanı) tek bir akışta birleşti.

**Not:** API anahtarı güvenlik gereği koda açık yazılmamalıdır; paylaşım öncesi
"BURAYA_KENDI_KEYIN" ile değiştirilmelidir.